# Hybrid Results Notebook

This notebook now prioritizes the **real Bologna grouped results** and keeps the synthetic branch only as a validation reference.

## What Is In This Notebook

1. path and artifact checks for real and synthetic runs
2. summary tables for real Stage 1 and real Stage 2
3. lon/lat map viewers for the real Bologna patch
4. an interactive lon/lat click viewer on InSAR velocity
5. a compact synthetic validation summary section

For the current real-data path, the active grouped state is:

- `Load_total = S0 + Ss + Sd + Sr`
- `Sg`

So the real interactive viewer currently shows those grouped channels, not the full five real layers yet.

Current interpretation:

- grouped Stage 1 is the current stable applied inversion result
- Stage 2 is shown as an experimental lag/mismatch learner and diagnostic layer
- the current Stage 2 runs do not yet outperform Stage 1 on the reconstructed full-map deformation metric

Conditional layered product:

- a practical five-layer posterior is also exported by redistributing the stable grouped `Load_total` posterior back into `S0, Ss, Sd, Sr` using W3RA layer shares, while keeping `Sg` directly from the grouped posterior


In [ ]:
%matplotlib widget

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

ROOT = Path('/home/ubuntu/work/insar_mcmc')

REAL_STAGE1_DIR = ROOT / 'outputs_stage1_bologna_real_full_grouped_quick'
REAL_STAGE2_DIR = ROOT / 'outputs_stage2_bologna_real_full_grouped_lagaware'
SYN_STAGE1_DIR = ROOT / 'outputs_stage1_pure'
SYN_STAGE2_DIR = ROOT / 'outputs_stage2_synthetic_smoke'
LAYERED_DIR = ROOT / 'outputs_layered_inference_from_grouped_full'

PATHS = {
    'real_stage1_npz': REAL_STAGE1_DIR / 'stage1_bologna_real_results.npz',
    'real_stage1_json': REAL_STAGE1_DIR / 'stage1_bologna_real_summary.json',
    'real_stage2_npz': REAL_STAGE2_DIR / 'stage2_bologna_real_results.npz',
    'real_stage2_json': REAL_STAGE2_DIR / 'stage2_bologna_real_summary.json',
    'syn_stage1_json': SYN_STAGE1_DIR / 'stage1_pure_synthetic_summary.json',
    'syn_stage2_json': SYN_STAGE2_DIR / 'stage2_residual_summary.json',
    'layered_npz': LAYERED_DIR / 'layered_inference_from_grouped.npz',
    'layered_json': LAYERED_DIR / 'layered_inference_from_grouped_summary.json',
}

for name, path in PATHS.items():
    print(f'{name}: {path} | exists={path.exists()}')


In [ ]:
def load_json(path):
    with open(path) as f:
        return json.load(f)

real_stage1_summary = load_json(PATHS['real_stage1_json']) if PATHS['real_stage1_json'].exists() else None
real_stage2_summary = load_json(PATHS['real_stage2_json']) if PATHS['real_stage2_json'].exists() else None
syn_stage1_summary = load_json(PATHS['syn_stage1_json']) if PATHS['syn_stage1_json'].exists() else None
syn_stage2_summary = load_json(PATHS['syn_stage2_json']) if PATHS['syn_stage2_json'].exists() else None
layered_summary = load_json(PATHS['layered_json']) if PATHS['layered_json'].exists() else None

real_stage1 = np.load(PATHS['real_stage1_npz']) if PATHS['real_stage1_npz'].exists() else None
real_stage2 = np.load(PATHS['real_stage2_npz']) if PATHS['real_stage2_npz'].exists() else None
layered_data = np.load(PATHS['layered_npz']) if PATHS['layered_npz'].exists() else None

if real_stage1 is not None:
    print('Real Stage 1 keys:', sorted(real_stage1.files))
if real_stage2 is not None:
    print('Real Stage 2 keys:', sorted(real_stage2.files))
if layered_data is not None:
    print('Layered product keys:', sorted(layered_data.files))


In [ ]:
def flat_table(d, key_name='metric', value_name='value'):
    return pd.DataFrame([{key_name: k, value_name: v} for k, v in d.items()])

if real_stage1_summary is not None:
    print('Real Stage 1 observation fit')
    display(pd.DataFrame([real_stage1_summary['observation_fit']]).round(6))
    print('Real Stage 1 theta summary')
    display(pd.DataFrame(real_stage1_summary['posterior']['theta_summary']).T.round(4))

if real_stage2_summary is not None:
    print('Real Stage 2 deformation-fit comparison on held-out tile windows')
    display(pd.DataFrame([real_stage2_summary['test_metrics']]).round(6))
    print('Real Stage 2 deformation-fit comparison on the reconstructed full Bologna map')
    display(pd.DataFrame([real_stage2_summary['full_map_metrics']]).round(6))
    print('Interpretation: Stage 1 remains the current applied estimator; Stage 2 is currently better viewed as a lag/mismatch diagnostic module.')


if layered_summary is not None:
    print('Layered inference consistency')
    display(pd.DataFrame([layered_summary['consistency']]).round(6))

if syn_stage1_summary is not None and syn_stage2_summary is not None:
    print('Synthetic validation anchors')
    syn_rows = [
        {'case': 'Stage1 clean TWS', **syn_stage1_summary['derived_state_metrics']['TWS']},
        {'case': 'Stage1 clean Sg', **syn_stage1_summary['state_metrics']['Sg']},
        {'case': 'Stage2 final deformation',
         'r2': syn_stage2_summary['test_metrics'].get('disp_final_r2', np.nan),
         'corr': syn_stage2_summary['test_metrics'].get('disp_final_corr', np.nan),
         'rmse': syn_stage2_summary['test_metrics'].get('disp_final_rmse', np.nan)},
    ]
    display(pd.DataFrame(syn_rows).round(4))


In [ ]:
FIELD_NAMES = real_stage1['field_names'].tolist() if real_stage1 is not None else ['Load_total', 'Sg']

def robust_limits(a, pct=99):
    vals = np.asarray(a)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return (-1.0, 1.0)
    lim = float(np.nanpercentile(np.abs(vals), pct))
    lim = max(lim, 1e-6)
    return (-lim, lim)


def linear_trend_map(arr_tyx):
    t = np.arange(arr_tyx.shape[0], dtype=np.float32)
    t = t - t.mean()
    denom = np.sum(t ** 2)
    flat = arr_tyx.reshape(arr_tyx.shape[0], -1)
    slopes = (t[:, None] * flat).sum(axis=0) / denom
    return slopes.reshape(arr_tyx.shape[1:])


def plot_lonlat(ax, lon, lat, field, title, cmap='RdBu', symmetric=True):
    if symmetric:
        vmin, vmax = robust_limits(field)
    else:
        vals = field[np.isfinite(field)]
        vmin = float(np.nanpercentile(vals, 1)) if vals.size else 0.0
        vmax = float(np.nanpercentile(vals, 99)) if vals.size else 1.0
    mesh = ax.pcolormesh(lon, lat, field, shading='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.colorbar(mesh, ax=ax, shrink=0.8)
    return mesh


if real_stage1 is not None:
    real_bundle = {
        'lat': real_stage1['lat'],
        'lon': real_stage1['lon'],
        'time': real_stage1['time'],
        'x_prior': real_stage1['x_prior'],
        'y_obs': real_stage1['y_obs'],
        'd_prior': real_stage1['y_pred'],
    }
    if real_stage2 is not None:
        real_bundle['x_final'] = real_stage2['x_final']
        real_bundle['d_final'] = real_stage2['d_final']

if layered_data is not None:
    layered_bundle = {
        'x_layered': layered_data['x_layered'],
        'load_total': layered_data['load_total'],
        'sg': layered_data['sg'],
        'tws': layered_data['tws'],
        'lat': layered_data['lat'],
        'lon': layered_data['lon'],
        'time': layered_data['time'],
        'field_names': layered_data['field_names'].tolist(),
    }


In [ ]:
if real_stage1 is not None:
    lon = real_bundle['lon']
    lat = real_bundle['lat']
    t_default = min(10, real_bundle['y_obs'].shape[0] - 1)

    source_dd = widgets.Dropdown(options=['prior', 'final'], value='final' if 'x_final' in real_bundle else 'prior', description='Source')
    field_dd = widgets.Dropdown(options=FIELD_NAMES + ['Observed_def', 'Prior_def', 'Final_def', 'Def_residual'], value=FIELD_NAMES[-1], description='Field')
    time_sl = widgets.IntSlider(value=t_default, min=0, max=real_bundle['y_obs'].shape[0] - 1, step=1, description='Time')
    out_maps = widgets.Output()

    def get_real_field(source, field, tidx):
        if field in FIELD_NAMES:
            idx = FIELD_NAMES.index(field)
            if source == 'prior':
                return real_bundle['x_prior'][tidx, idx]
            return real_bundle.get('x_final', real_bundle['x_prior'])[tidx, idx]
        if field == 'Observed_def':
            return real_bundle['y_obs'][tidx]
        if field == 'Prior_def':
            return real_bundle['d_prior'][tidx]
        if field == 'Final_def':
            return real_bundle.get('d_final', real_bundle['d_prior'])[tidx]
        if field == 'Def_residual':
            return real_bundle['y_obs'][tidx] - (real_bundle.get('d_final', real_bundle['d_prior'])[tidx] if source == 'final' else real_bundle['d_prior'][tidx])
        raise ValueError(field)

    def refresh_real_maps(*_):
        with out_maps:
            out_maps.clear_output(wait=True)
            field = get_real_field(source_dd.value, field_dd.value, time_sl.value)
            fig, ax = plt.subplots(1, 1, figsize=(6, 5))
            plot_lonlat(ax, lon, lat, field, f'{field_dd.value} | {source_dd.value} | t={time_sl.value}')
            plt.show()

    for w in [source_dd, field_dd, time_sl]:
        w.observe(refresh_real_maps, names='value')
    display(widgets.HBox([source_dd, field_dd]), time_sl, out_maps)
    refresh_real_maps()
else:
    print('Real Stage 1 output is not available.')


In [ ]:
if 'layered_bundle' in globals():
    layer_dd = widgets.Dropdown(options=layered_bundle['field_names'] + ['TWS'], value='Sg', description='Layer')
    t_layer = widgets.IntSlider(value=min(10, layered_bundle['x_layered'].shape[0] - 1), min=0, max=layered_bundle['x_layered'].shape[0] - 1, step=1, description='Time')
    out_layered = widgets.Output()

    def refresh_layered(*_):
        with out_layered:
            out_layered.clear_output(wait=True)
            field = layer_dd.value
            tidx = t_layer.value
            if field == 'TWS':
                arr = layered_bundle['tws'][tidx]
            else:
                arr = layered_bundle['x_layered'][tidx, layered_bundle['field_names'].index(field)]
            fig, ax = plt.subplots(1, 1, figsize=(6, 5))
            plot_lonlat(ax, layered_bundle['lon'], layered_bundle['lat'], arr, f'Layered inference | {field} | t={tidx}')
            plt.show()

    layer_dd.observe(refresh_layered, names='value')
    t_layer.observe(refresh_layered, names='value')
    display(widgets.HBox([layer_dd]), t_layer, out_layered)
    refresh_layered()
else:
    print('Layered inference export is not available.')


In [ ]:
if real_stage1 is not None:
    obs_vel = linear_trend_map(real_bundle['y_obs'])
    prior_vel = linear_trend_map(real_bundle['d_prior'])
    final_vel = linear_trend_map(real_bundle.get('d_final', real_bundle['d_prior']))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    plot_lonlat(axes[0], lon, lat, obs_vel, 'Observed InSAR Velocity')
    plot_lonlat(axes[1], lon, lat, final_vel - prior_vel, 'Stage 2 minus Stage 1 Velocity')
    plt.show()


In [ ]:
if real_stage1 is not None:
    click_out = widgets.Output()
    fig, ax = plt.subplots(figsize=(6, 5))
    plot_lonlat(ax, lon, lat, linear_trend_map(real_bundle['y_obs']), 'Click a point on Observed InSAR Velocity')

    def nearest_ij(xlon, ylat):
        dist2 = (lon - xlon) ** 2 + (lat - ylat) ** 2
        return np.unravel_index(np.nanargmin(dist2), dist2.shape)

    def on_click(event):
        if event.inaxes is not ax or event.xdata is None or event.ydata is None:
            return
        j, i = nearest_ij(event.xdata, event.ydata)
        with click_out:
            click_out.clear_output(wait=True)
            fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
            tt = np.arange(real_bundle['y_obs'].shape[0])
            if 'layered_bundle' in globals():
                for k, name in enumerate(layered_bundle['field_names']):
                    axes2[0].plot(tt, layered_bundle['x_layered'][:, k, j, i], label=name)
                axes2[0].set_title(f'Layered time series @ lon={lon[j, i]:.4f}, lat={lat[j, i]:.4f}')
            else:
                axes2[0].plot(tt, real_bundle['x_prior'][:, 0, j, i], label='Load_total prior')
                axes2[0].plot(tt, real_bundle['x_prior'][:, 1, j, i], label='Sg prior')
                if 'x_final' in real_bundle:
                    axes2[0].plot(tt, real_bundle['x_final'][:, 0, j, i], '--', label='Load_total final')
                    axes2[0].plot(tt, real_bundle['x_final'][:, 1, j, i], '--', label='Sg final')
                axes2[0].set_title(f'Grouped state time series @ lon={lon[j, i]:.4f}, lat={lat[j, i]:.4f}')
            axes2[0].set_xlabel('Time index')
            axes2[0].legend()

            axes2[1].plot(tt, real_bundle['y_obs'][:, j, i], label='Observed def')
            axes2[1].plot(tt, real_bundle['d_prior'][:, j, i], label='Prior def')
            if 'd_final' in real_bundle:
                axes2[1].plot(tt, real_bundle['d_final'][:, j, i], '--', label='Final def')
            axes2[1].set_title('Deformation time series')
            axes2[1].set_xlabel('Time index')
            axes2[1].legend()
            plt.show()

    cid = fig.canvas.mpl_connect('button_press_event', on_click)
    display(click_out)
else:
    print('Real Stage 1 output is not available.')


In [ ]:
if real_stage1_summary is not None and real_stage2_summary is not None:
    compare_rows = []
    for scope, block in [('Held-out tile windows', real_stage2_summary['test_metrics']), ('Full reconstructed map', real_stage2_summary['full_map_metrics'])]:
        compare_rows.extend([
            {
                'scope': scope,
                'metric': 'RMSE',
                'stage1_prior': block['disp_prior_rmse'],
                'stage2_final': block['disp_final_rmse'],
                'delta': block['disp_final_rmse'] - block['disp_prior_rmse'],
            },
            {
                'scope': scope,
                'metric': 'Correlation',
                'stage1_prior': block['disp_prior_corr'],
                'stage2_final': block['disp_final_corr'],
                'delta': block['disp_final_corr'] - block['disp_prior_corr'],
            },
            {
                'scope': scope,
                'metric': 'R2',
                'stage1_prior': block['disp_prior_r2'],
                'stage2_final': block['disp_final_r2'],
                'delta': block['disp_final_r2'] - block['disp_prior_r2'],
            },
        ])
    compare_df = pd.DataFrame(compare_rows)
    display(compare_df.round(6))


## Notes

- The current real-data notebook section is intentionally grouped and uses `Load_total + Sg` because the first full five-layer real Stage 1 tests were too ill-conditioned.
- The synthetic branch is still useful as a validation reference, but the real Bologna full subset is now the main results view.
- The current Stage 2 run is the stronger variant that uses Stage 1 `theta` context, MintPy coherence context, and a Stage-1-confidence anchor.
- The current Stage 2 run is a lag-aware, bounded residual variant that also sees the Stage 1 deformation-mismatch history through time.
- The applied takeaway is still conservative: grouped Stage 1 is the current stable inversion result, while Stage 2 remains an exploratory temporal-diagnostic extension.
- All real maps in this notebook are drawn directly on the saved `lon/lat` arrays rather than pixel indices.


## Grouped Balanced Kalman

This section shows the current **best real Bologna Stage 1** result: the grouped balanced multisensor Kalman model on the corrected 2025 MintPy overlap, using:

- InSAR deformation anomalies
- GRACE regional anomalies
- refreshed SMAP surface soil moisture

Interpretation:

- this is the first real-data path that stays on physically sane grouped magnitudes
- it is still **grouped** (`ShallowLoad`, `DeepLoad`, `Groundwater`), not full layered inversion
- the main question is whether the grouped posterior is both physically sane and externally supported by independent well data

In [ ]:
KALMAN_REGIONAL_DIR = ROOT / 'outputs_stage1_bologna_multisensor_kalman_overlap2025_smaprefresh'
KALMAN_TILED_DIR = ROOT / 'outputs_stage1_bologna_multisensor_kalman_tiled_overlap2025_smaprefresh'
KALMAN_SWOT_DIR = ROOT / 'outputs_stage1_bologna_multisensor_kalman_overlap2025_swot'

KALMAN_PATHS = {
    'regional_json': KALMAN_REGIONAL_DIR / 'stage1_bologna_multisensor_kalman_summary.json',
    'regional_npz': KALMAN_REGIONAL_DIR / 'stage1_bologna_multisensor_kalman_results.npz',
    'regional_csv': KALMAN_REGIONAL_DIR / 'stage1_bologna_multisensor_kalman_timeseries.csv',
    'tiled_json': KALMAN_TILED_DIR / 'stage1_bologna_multisensor_kalman_tiled_summary.json',
    'tiled_npz': KALMAN_TILED_DIR / 'stage1_bologna_multisensor_kalman_tiled_results.npz',
    'swot_json': KALMAN_SWOT_DIR / 'stage1_bologna_multisensor_kalman_summary.json',
}

for name, path in KALMAN_PATHS.items():
    print(f'{name}: {path} | exists={path.exists()}')

kalman_regional_summary = load_json(KALMAN_PATHS['regional_json']) if KALMAN_PATHS['regional_json'].exists() else None
kalman_tiled_summary = load_json(KALMAN_PATHS['tiled_json']) if KALMAN_PATHS['tiled_json'].exists() else None
kalman_swot_summary = load_json(KALMAN_PATHS['swot_json']) if KALMAN_PATHS['swot_json'].exists() else None
kalman_regional = np.load(KALMAN_PATHS['regional_npz']) if KALMAN_PATHS['regional_npz'].exists() else None
kalman_tiled = np.load(KALMAN_PATHS['tiled_npz']) if KALMAN_PATHS['tiled_npz'].exists() else None

if kalman_regional is not None:
    print('Kalman regional keys:', sorted(kalman_regional.files))
if kalman_tiled is not None:
    print('Kalman tiled keys:', sorted(kalman_tiled.files))

In [ ]:
if kalman_regional_summary is not None:
    print('Regional grouped Kalman metrics')
    rows = []
    for sensor in ['insar', 'grace', 'smap']:
        rows.append({
            'sensor': sensor,
            'n_obs': kalman_regional_summary['metrics'][f'{sensor}_post']['n'],
            'prior_r2': kalman_regional_summary['metrics'][f'{sensor}_prior']['r2'],
            'post_r2': kalman_regional_summary['metrics'][f'{sensor}_post']['r2'],
            'delta_r2': kalman_regional_summary['metrics'][f'{sensor}_post']['r2'] - kalman_regional_summary['metrics'][f'{sensor}_prior']['r2'],
            'prior_corr': kalman_regional_summary['metrics'][f'{sensor}_prior']['corr'],
            'post_corr': kalman_regional_summary['metrics'][f'{sensor}_post']['corr'],
            'prior_rmse': kalman_regional_summary['metrics'][f'{sensor}_prior']['rmse'],
            'post_rmse': kalman_regional_summary['metrics'][f'{sensor}_post']['rmse'],
        })
    display(pd.DataFrame(rows).round(4))

if kalman_swot_summary is not None:
    print('SWOT-added comparison')
    swot_rows = []
    for sensor in ['swot_river', 'swot_lake']:
        swot_rows.append({
            'sensor': sensor,
            'n_obs': kalman_swot_summary['metrics'][f'{sensor}_post']['n'],
            'prior_r2': kalman_swot_summary['metrics'][f'{sensor}_prior']['r2'],
            'post_r2': kalman_swot_summary['metrics'][f'{sensor}_post']['r2'],
            'post_corr': kalman_swot_summary['metrics'][f'{sensor}_post']['corr'],
            'post_rmse': kalman_swot_summary['metrics'][f'{sensor}_post']['rmse'],
        })
    display(pd.DataFrame(swot_rows).round(4))

if kalman_tiled_summary is not None:
    print('Tiled grouped Kalman fit and magnitude summary')
    display(pd.DataFrame([kalman_tiled_summary['metrics']['tile_insar_post']]).round(4))
    mag_df = pd.DataFrame(kalman_tiled_summary['magnitude']).T
    display(mag_df.round(4))

try:
    import xarray as xr
    w = xr.open_dataset('/home/ubuntu/work/insar_mcmc/outputs_bologna_2025_overlap/w3ra_on_mintpy2025_overlap_anom.nc')
    raw_rows = []
    grouped_raw = {
        'ShallowLoad': (w['S0'] + w['Ss']).values,
        'DeepLoad': (w['Sd'] + w['Sr']).values,
        'Groundwater': w['Sg'].values,
    }
    for name, arr in grouped_raw.items():
        raw_rows.append({
            'state': name,
            'raw_abs_max_mm': float(np.nanmax(np.abs(arr))),
            'raw_std_mm': float(np.nanstd(arr)),
            'posterior_abs_max_mm': kalman_tiled_summary['magnitude'][name]['x_abs_max'] if kalman_tiled_summary is not None else np.nan,
        })
    raw_df = pd.DataFrame(raw_rows)
    raw_df['posterior_to_raw_absmax_ratio'] = raw_df['posterior_abs_max_mm'] / raw_df['raw_abs_max_mm']
    print('Sanity check against grouped W3RA anomalies on the corrected overlap')
    display(raw_df.round(4))
    w.close()
except Exception as exc:
    print('Could not compute raw W3RA sanity table:', exc)

In [ ]:
if kalman_tiled is not None:
    kalman_state_names = kalman_tiled['state_names'].tolist()
    kalman_lon = kalman_tiled['lon_tiles']
    kalman_lat = kalman_tiled['lat_tiles']
    kalman_field_dd = widgets.Dropdown(options=kalman_state_names + ['theta_ShallowLoad', 'theta_DeepLoad', 'theta_Groundwater', 'insar_obs', 'insar_post'], value='Groundwater', description='Field')
    kalman_time_sl = widgets.IntSlider(value=min(10, kalman_tiled['x_tiles'].shape[0] - 1), min=0, max=kalman_tiled['x_tiles'].shape[0] - 1, step=1, description='Time')
    kalman_out = widgets.Output()

    def get_kalman_field(field, tidx):
        if field in kalman_state_names:
            return kalman_tiled['x_tiles'][tidx, kalman_state_names.index(field)]
        if field.startswith('theta_'):
            name = field.split('_', 1)[1]
            return kalman_tiled['theta_tiles'][tidx, kalman_state_names.index(name)]
        if field == 'insar_obs':
            return kalman_tiled['y_obs_tiles'][tidx]
        if field == 'insar_post':
            return kalman_tiled['y_post_tiles'][tidx]
        raise ValueError(field)

    def refresh_kalman(*_):
        with kalman_out:
            kalman_out.clear_output(wait=True)
            arr = get_kalman_field(kalman_field_dd.value, kalman_time_sl.value)
            fig, ax = plt.subplots(1, 1, figsize=(6, 5))
            plot_lonlat(ax, kalman_lon, kalman_lat, arr, f'Kalman grouped | {kalman_field_dd.value} | t={kalman_time_sl.value}')
            plt.show()

    kalman_field_dd.observe(refresh_kalman, names='value')
    kalman_time_sl.observe(refresh_kalman, names='value')
    display(widgets.HBox([kalman_field_dd]), kalman_time_sl, kalman_out)
    refresh_kalman()
else:
    print('Tiled grouped Kalman output is not available.')


In [ ]:
if kalman_regional is not None:
    kalman_ts = pd.read_csv(KALMAN_PATHS['regional_csv'])
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)

    axes[0].plot(pd.to_datetime(kalman_ts['time']), kalman_ts['y_insar'], label='InSAR obs')
    axes[0].plot(pd.to_datetime(kalman_ts['time']), kalman_ts['y_insar_post'], label='InSAR post')
    axes[0].plot(pd.to_datetime(kalman_ts['time']), kalman_ts['y_grace'], label='GRACE obs')
    axes[0].plot(pd.to_datetime(kalman_ts['time']), kalman_ts['y_grace_post'], label='GRACE post')
    axes[0].plot(pd.to_datetime(kalman_ts['time']), kalman_ts['y_smap'], label='SMAP obs')
    axes[0].plot(pd.to_datetime(kalman_ts['time']), kalman_ts['y_smap_post'], label='SMAP post')
    axes[0].set_title('Regional multisensor fit')
    axes[0].legend(ncol=3, fontsize=8)

    axes[1].plot(pd.to_datetime(kalman_ts['time']), kalman_ts['x_shallow'], label='ShallowLoad')
    axes[1].plot(pd.to_datetime(kalman_ts['time']), kalman_ts['x_deep'], label='DeepLoad')
    axes[1].plot(pd.to_datetime(kalman_ts['time']), kalman_ts['x_groundwater'], label='Groundwater')
    axes[1].set_title('Posterior grouped states')
    axes[1].legend()
    plt.show()

    print('Simple verdict')
    verdict = pd.DataFrame([{
        'question': 'Do grouped posterior magnitudes stay below 100 mm?',
        'answer': bool(max(v['x_abs_max'] for v in kalman_tiled_summary['magnitude'].values()) < 100.0) if kalman_tiled_summary is not None else None,
    }, {
        'question': 'Did regional InSAR R2 improve over prior?',
        'answer': bool(kalman_regional_summary['metrics']['insar_post']['r2'] > kalman_regional_summary['metrics']['insar_prior']['r2']) if kalman_regional_summary is not None else None,
    }, {
        'question': 'Did regional GRACE R2 improve over prior?',
        'answer': bool(kalman_regional_summary['metrics']['grace_post']['r2'] > kalman_regional_summary['metrics']['grace_prior']['r2']) if kalman_regional_summary is not None else None,
    }])
    display(verdict)
else:
    print('Regional grouped Kalman output is not available.')


## Wells Validation

This section compares the grouped Stage 1 `Groundwater` state against independent Bologna well observations.

Important interpretation:

- the model state is a grouped groundwater-storage-like signal
- the wells measure hydraulic head (`piezometry`) or depth to water
- so this is a **related-variable validation**, not a same-units one-to-one comparison
- the right checks are anomaly correlation, trend agreement, and temporal consistency

In [ ]:
WELL_VAL_DIR = ROOT / 'outputs_well_validation'
WELL_PATHS = {
    'summary_csv': WELL_VAL_DIR / 'well_groundwater_validation_summary.csv',
    'overview_json': WELL_VAL_DIR / 'well_groundwater_validation_overview.json',
    'by_depth_csv': WELL_VAL_DIR / 'well_groundwater_validation_by_depth.csv',
    'by_gwb_csv': WELL_VAL_DIR / 'well_groundwater_validation_by_gwb.csv',
    'trusted_gwb_csv': WELL_VAL_DIR / 'well_groundwater_validation_trusted_gwb.csv',
    'lag_state_csv': WELL_VAL_DIR / 'well_groundwater_validation_lag_state_counts.csv',
    'topstations_csv': WELL_VAL_DIR / 'well_groundwater_validation_topstations.csv',
    'trusted_stations_csv': WELL_VAL_DIR / 'well_groundwater_validation_trusted_stations.csv',
    'series_dir': WELL_VAL_DIR / 'station_series',
    'wells_long': ROOT / 'outputs_external_bologna_wells/processed/bologna_wells_long.csv',
}

for name, path in WELL_PATHS.items():
    print(f'{name}: {path} | exists={path.exists()}')

well_overview = load_json(WELL_PATHS['overview_json']) if WELL_PATHS['overview_json'].exists() else None
well_summary = pd.read_csv(WELL_PATHS['summary_csv']) if WELL_PATHS['summary_csv'].exists() else None
well_by_depth = pd.read_csv(WELL_PATHS['by_depth_csv']) if WELL_PATHS['by_depth_csv'].exists() else None
well_by_gwb = pd.read_csv(WELL_PATHS['by_gwb_csv']) if WELL_PATHS['by_gwb_csv'].exists() else None
well_trusted_gwb = pd.read_csv(WELL_PATHS['trusted_gwb_csv']) if WELL_PATHS['trusted_gwb_csv'].exists() else None
well_lag_state = pd.read_csv(WELL_PATHS['lag_state_csv']) if WELL_PATHS['lag_state_csv'].exists() else None
well_topstations = pd.read_csv(WELL_PATHS['topstations_csv']) if WELL_PATHS['topstations_csv'].exists() else None
well_trusted_stations = pd.read_csv(WELL_PATHS['trusted_stations_csv']) if WELL_PATHS['trusted_stations_csv'].exists() else None
well_long = pd.read_csv(WELL_PATHS['wells_long'], parse_dates=['date']) if WELL_PATHS['wells_long'].exists() else None

if well_summary is not None:
    print('Well validation rows:', len(well_summary))

In [ ]:
if well_overview is not None:
    print('Well validation overview')
    display(pd.DataFrame([well_overview]).round(4))

if well_by_depth is not None:
    print('Validation by well depth class')
    display(well_by_depth.round(4))

if well_trusted_gwb is not None:
    print('Trusted aquifer groups')
    display(well_trusted_gwb.round(4))
elif well_by_gwb is not None:
    print('Top aquifer groups by validation support')
    display(well_by_gwb.head(15).round(4))

if well_lag_state is not None:
    print('Best state / lag counts')
    display(well_lag_state)

if well_trusted_stations is not None:
    print('Trusted station panel')
    display(well_trusted_stations.head(25).round(4))
elif well_topstations is not None:
    print('Top station shortlist')
    display(well_topstations.head(20).round(4))

if well_summary is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    plot_df = well_trusted_stations if well_trusted_stations is not None and not well_trusted_stations.empty else well_summary
    sc = axes[0].scatter(plot_df['lon'], plot_df['lat'], c=plot_df['corr_anom'], s=30 + 3 * plot_df['n_matches'], cmap='viridis', vmin=-1, vmax=1)
    axes[0].set_xlabel('Longitude')
    axes[0].set_ylabel('Latitude')
    axes[0].set_title('Trusted well panel | color = anomaly correlation')
    plt.colorbar(sc, ax=axes[0], shrink=0.8, label='corr_anom')

    if well_by_depth is not None and not well_by_depth.empty:
        axes[1].bar(well_by_depth['depth_class'], well_by_depth['median_corr'])
        axes[1].set_title('Median correlation by depth class')
        axes[1].set_ylabel('median corr')
        axes[1].set_ylim(0, 1)
    else:
        axes[1].axis('off')
    plt.show()
else:
    print('Well validation outputs are not available.')

In [ ]:
if well_summary is not None and WELL_PATHS['series_dir'].exists():
    base_df = well_trusted_stations if 'well_trusted_stations' in globals() and well_trusted_stations is not None and not well_trusted_stations.empty else well_summary
    station_opts = base_df['station_code'].tolist()
    station_dd = widgets.Dropdown(options=station_opts, value=station_opts[0], description='Station')
    well_out = widgets.Output()

    fig, (axm, axt) = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    def render_station(code):
        row = base_df.loc[base_df['station_code'] == code].iloc[0]
        series_path = WELL_PATHS['series_dir'] / f"{code}_{row['measurement_type']}.csv"
        ser = pd.read_csv(series_path, parse_dates=['date'])

        axm.clear()
        axm.scatter(base_df['lon'], base_df['lat'], c=base_df['corr_anom'], cmap='viridis', vmin=-1, vmax=1, s=30 + 3 * base_df['n_matches'])
        axm.scatter([row['lon']], [row['lat']], marker='x', s=120, c='red')
        axm.set_xlabel('Longitude')
        axm.set_ylabel('Latitude')
        axm.set_title('Click a station or use dropdown')

        axt.clear()
        axt.plot(ser['date'], ser['model_anom_z'], label='Grouped Groundwater (z-anom)')
        axt.plot(ser['date'], ser['obs_anom_z'], label='Well head (z-anom)')
        axt.set_title(f"{code} | corr={row['corr_anom']:.3f} | n={int(row['n_matches'])}")
        axt.legend()

        with well_out:
            well_out.clear_output(wait=True)
            display(pd.DataFrame([row]).round(4))

        fig.canvas.draw_idle()

    def on_change(change):
        if change['name'] == 'value' and change['new'] is not None:
            render_station(change['new'])

    def on_click(event):
        if event.inaxes != axm or event.xdata is None or event.ydata is None:
            return
        d2 = (base_df['lon'] - event.xdata) ** 2 + (base_df['lat'] - event.ydata) ** 2
        idx = d2.idxmin()
        station_dd.value = base_df.loc[idx, 'station_code']

    station_dd.observe(on_change, names='value')
    fig.canvas.mpl_connect('button_press_event', on_click)
    display(station_dd, fig.canvas, well_out)
    render_station(station_dd.value)
else:
    print('Interactive well validation view is not available.')

## Kalman Notes

- The grouped balanced Kalman path is now the strongest real-data direction.
- The current best run uses the corrected MintPy/W3RA overlap and the refreshed SMAP time series.
- Posterior grouped magnitudes stay in a tens-of-mm range rather than exploding to unphysical values.
- Regional fit improves strongly for InSAR, GRACE, and SMAP.
- Adding SWOT as a direct observation operator is technically possible, but it does not help much yet because only a few matched SWOT dates are available.
- This is still not a full layered hydrology result, but it is a credible grouped posterior that can now be checked against independent groundwater wells.

## Validation Figures

These are the compact exported figures for the trusted wells validation panel.

In [ ]:
from IPython.display import Image as IPyImage, display as ipy_display
FIG_DIR = ROOT / 'outputs_well_validation' / 'figures'
FIGS = [
    'well_validation_summary_panel.png',
    'trusted_wells_map.png',
    'trusted_aquifer_groups.png',
    'lag_histogram_by_state.png',
    'depth_class_summary.png',
    'top6_station_timeseries.png',
]
for name in FIGS:
    path = FIG_DIR / name
    print(path, '| exists=', path.exists())
    if path.exists():
        ipy_display(IPyImage(filename=str(path)))

## Emilia-Romagna Wells Explorer

This interactive viewer uses the full Emilia-Romagna well network metadata and time series. Click a station on the map, or use the controls, to inspect the station history. The selected station ID is annotated on the map.


In [ ]:
ER_WELLS_PATH = ROOT / 'outputs_external_bologna_wells' / 'processed' / 'all_wells_long.csv'
ER_META_PATH = ROOT / 'outputs_external_bologna_wells' / 'processed' / 'station_metadata_combined.csv'

er_wells = pd.read_csv(ER_WELLS_PATH, parse_dates=['date']) if ER_WELLS_PATH.exists() else None
er_meta = pd.read_csv(ER_META_PATH) if ER_META_PATH.exists() else None

if er_wells is not None:
    er_station_summary = (
        er_wells.groupby('station_code')
        .agg(
            n_obs=('date', 'size'),
            first_date=('date', 'min'),
            last_date=('date', 'max'),
            n_piezometry=('piezometry_m', lambda s: s.notna().sum()),
            n_depth_to_water=('depth_to_water_m', lambda s: s.notna().sum()),
            measurement_type=('measurement_type', lambda s: ','.join(sorted(set(map(str, s.dropna()))))),
        )
        .reset_index()
    )
    if er_meta is not None:
        er_station_summary = er_station_summary.merge(er_meta.drop_duplicates('station_code'), on='station_code', how='left')
    print('Full Emilia-Romagna station summary')
    display(er_station_summary[['station_code','province','municipality','gwb_name','measurement_type','n_obs','first_date','last_date','lon','lat']].head(20))
else:
    print('Full Emilia-Romagna wells table not available.')


In [ ]:
if er_wells is not None and er_meta is not None:
    er_station_summary = er_station_summary.sort_values(['province','station_code']).reset_index(drop=True)
    province_opts = ['ALL'] + sorted([p for p in er_station_summary['province'].dropna().unique().tolist()])
    province_dd = widgets.Dropdown(options=province_opts, value='ALL', description='Province')
    station_dd = widgets.Dropdown(description='Station')
    variable_dd = widgets.Dropdown(options=[('Piezometry head', 'piezometry_m'), ('Depth to water', 'depth_to_water_m')], value='piezometry_m', description='Series')
    er_out = widgets.Output()

    fig, (ax_map, ax_ts) = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    def current_station_table():
        if province_dd.value == 'ALL':
            return er_station_summary.copy()
        return er_station_summary.loc[er_station_summary['province'] == province_dd.value].copy()

    def refresh_station_options(preferred=None):
        station_table = current_station_table()
        opts = station_table['station_code'].tolist()
        station_dd.options = opts
        if not opts:
            station_dd.value = None
        elif preferred in opts:
            station_dd.value = preferred
        else:
            station_dd.value = opts[0]

    def render_station(code):
        station_table = current_station_table()
        if code is None or station_table.empty:
            return
        row = station_table.loc[station_table['station_code'] == code].iloc[0]
        ser = er_wells.loc[er_wells['station_code'] == code].sort_values('date').copy()
        ycol = variable_dd.value
        ylab = 'Piezometry head (m)' if ycol == 'piezometry_m' else 'Depth to water (m)'

        ax_map.clear()
        scatter = ax_map.scatter(station_table['lon'], station_table['lat'], s=10 + 0.05 * station_table['n_obs'], c=station_table['n_obs'], cmap='viridis', alpha=0.8)
        ax_map.scatter([row['lon']], [row['lat']], marker='x', s=140, c='red')
        ax_map.annotate(code, (row['lon'], row['lat']), xytext=(6, 6), textcoords='offset points', color='darkred', fontsize=9, fontweight='bold')
        ax_map.set_xlabel('Longitude')
        ax_map.set_ylabel('Latitude')
        ax_map.set_title(f'Emilia-Romagna wells | province={province_dd.value}')
        if not hasattr(render_station, '_cbar') or render_station._cbar is None:
            render_station._cbar = fig.colorbar(scatter, ax=ax_map, shrink=0.8, label='Observation count')
        else:
            render_station._cbar.update_normal(scatter)

        ax_ts.clear()
        if ser[ycol].notna().any():
            ax_ts.plot(ser['date'], ser[ycol], lw=1.5)
            ax_ts.set_ylabel(ylab)
        else:
            ax_ts.text(0.5, 0.5, f'No {ylab.lower()} available', ha='center', va='center', transform=ax_ts.transAxes)
        ax_ts.set_title(f"{code} | {row.get('municipality', 'NA')} | {row.get('gwb_name', 'NA')}")
        ax_ts.tick_params(axis='x', rotation=25)

        with er_out:
            er_out.clear_output(wait=True)
            display(pd.DataFrame([row])[['station_code','province','municipality','gwb_name','well_depth_m','measurement_type','n_obs','first_date','last_date','lon','lat']].round(4))

        fig.canvas.draw_idle()

    def on_station_change(change):
        if change['name'] == 'value' and change['new'] is not None:
            render_station(change['new'])

    def on_province_change(change):
        if change['name'] == 'value':
            refresh_station_options()
            if station_dd.value is not None:
                render_station(station_dd.value)

    def on_variable_change(change):
        if change['name'] == 'value' and station_dd.value is not None:
            render_station(station_dd.value)

    def on_click(event):
        station_table = current_station_table()
        if event.inaxes != ax_map or event.xdata is None or event.ydata is None or station_table.empty:
            return
        d2 = (station_table['lon'] - event.xdata) ** 2 + (station_table['lat'] - event.ydata) ** 2
        idx = d2.idxmin()
        station_dd.value = station_table.loc[idx, 'station_code']

    refresh_station_options()
    station_dd.observe(on_station_change, names='value')
    province_dd.observe(on_province_change, names='value')
    variable_dd.observe(on_variable_change, names='value')
    fig.canvas.mpl_connect('button_press_event', on_click)
    display(widgets.HBox([province_dd, station_dd, variable_dd]), fig.canvas, er_out)
    if station_dd.value is not None:
        render_station(station_dd.value)
else:
    print('Interactive Emilia-Romagna wells explorer is not available.')
